# TOPOTEX Model Inspector — 最新模型验收

**唯一正式模型**：Image+Mesh → Z_F [F,384] → **Factorized Dense UV Query
Encoder**（face addr Linear(384,96) + bary MLP(27→96→96) + LN 融合 →
dense [96,256,256] → Conv2d(96,384,8,8) → tokens [1024,384]）→ Global UV
Query Attention（d4/h8, K=V=Z_F）→ UV condition [64,256,256] → Flow
Matching → texture [3,256,256]。所有 shape 来自 live forward，绝无手写。

In [ ]:
import json, os, sys, types
from pathlib import Path
p = Path.cwd()
PROJECT_ROOT = next(c for c in (p, *p.parents) if (c / "configs").exists())
sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------ configuration
DATA = Path(os.environ.get("TOPOTEX_DATA", "/root/youjiaZhang/topotex_data"))
RUN_ROOT = Path(os.environ.get("TOPOTEX_RUN_ROOT", DATA / "runs"))
RUN_DIR = Path(os.environ.get("INSPECT_RUN", RUN_ROOT / "fm_accept_overfit1"))
CHECKPOINT = "latest"       # "latest" | explicit path
DATASET_ROOT = Path(os.environ.get("TOPOTEX_DATASET_ROOT", DATA / "dataset"))
SPLIT_PATH = Path(os.environ.get("TOPOTEX_SPLIT", DATA / "object_split.json"))
SUBSET = os.environ.get("INSPECT_SUBSET", "train")   # "train" | "test"
SAMPLE_ID = os.environ.get("INSPECT_SAMPLE") or None
RANDOM_SAMPLE = True
RANDOM_SEED = 20260727
QUERY = "native"            # native | xatlas | blender_smart | partial
MODE = os.environ.get("INSPECT_MODE", "quick")       # quick | full
DEVICE = "cuda:0"

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC",
                                          "Noto Sans CJK JP", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from topotex import TopoTexDataset, TopoTexPipeline

split = json.loads(SPLIT_PATH.read_text())
TRAIN, TEST = split["train"], split["val"]
Q2DIR = {"native": "uv_000", "xatlas": "uv_001",
         "blender_smart": "uv_test", "partial": "uv_002"}
def resolve_ckpt(run_dir):
    for name in ("ckpt.pt", "ckpt_final.pt"):
        if (Path(run_dir) / name).exists():
            return Path(run_dir) / name
    raise FileNotFoundError(f"no checkpoint under {run_dir}")

ck_path = resolve_ckpt(RUN_DIR) if CHECKPOINT == "latest" else Path(CHECKPOINT)
pipe = TopoTexPipeline.from_checkpoint(ck_path, DEVICE)
ck = pipe.checkpoint
model = pipe.model
print("run:", RUN_DIR.name, "| step", ck.get("global_step"),
      "| Dq =", model.conditioner.decoder.texel_dim)
print("params: conditioner %.2fM + dit %.2fM" % (
    sum(x.numel() for x in model.conditioner.parameters()) / 1e6,
    sum(x.numel() for x in model.dit.parameters()) / 1e6))
trained = set(ck["samples"])
assert not (trained & set(TEST)), "TEST OBJECT LEAKED INTO TRAINING"
print("train/test leakage check: PASS (no trained id in unseen test)")

## Training Monitor（容忍在写文件/半行/原子发布）

In [ ]:
def read_jsonl_tolerant(p):
    rows = []
    try:
        for l in open(p):
            try:
                rows.append(json.loads(l))
            except json.JSONDecodeError:
                break
    except FileNotFoundError:
        pass
    return rows

rows = read_jsonl_tolerant(RUN_DIR / "metrics.jsonl")
cfg = ck["config"]
if rows:
    last = rows[-1]
    tot = cfg.get("steps", last["step"])
    print(f"step {last['step']} / {tot} ({100*last['step']/max(tot,1):.1f}%)"
          f" | loss {last['loss']:.4f} ema {last['loss_ema']:.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot([r["step"] for r in rows], [r["loss_ema"] for r in rows], lw=1)
    axes[0].set_yscale("log"); axes[0].set_title("loss EMA", fontsize=9)
prof = {}
try:
    prof = json.loads((RUN_DIR / "training_profile.json").read_text())
except Exception as e:
    print("profile unavailable:", e)
if prof.get("rows"):
    pr = prof["rows"]
    axes[1].plot([r["step"] for r in pr], [r["meshes_per_sec"] for r in pr], lw=1)
    axes[1].set_title("mesh exposures/sec", fontsize=9)
    last_p = pr[-1]
    print(f"{last_p['steps_per_sec']} st/s | {last_p['meshes_per_sec']} mesh/s | "
          f"util {last_p['gpu_util']}% | {last_p['memory_mb']/1024:.1f} GB | {last_p['power_w']} W")
plt.tight_layout(); plt.show()
print("checkpoints:", sorted(x.name for x in RUN_DIR.glob("ckpt*.pt")))
print("provenance: split_sha", str(cfg.get("split_sha256"))[:16],
      "| encoder", cfg.get("uv_query_encoder"), "| world", cfg.get("world_size"))

## 真实中间结果（live forward + 逐项断言）

In [ ]:
pool = TRAIN if SUBSET == "train" else TEST
rng = np.random.default_rng(RANDOM_SEED)
# overfit1 pins its own training sample ONLY for the default train
# subset — an explicit SUBSET="test" always draws from unseen objects
pin_overfit = RUN_DIR.name == "fm_accept_overfit1" and SUBSET == "train"
sid = SAMPLE_ID or (ck["samples"][0] if pin_overfit
                    else pool[int(rng.integers(len(pool)))])
it = TopoTexDataset(DATASET_ROOT, [sid], device=DEVICE)[0]
print("sample:", sid, "| split:", "train" if sid in set(TRAIN) else "TEST")
Fn = len(it["mesh"]["faces"])
dec = model.conditioner.decoder
caps = {}
hooks = [
    dec.face_proj.register_forward_hook(lambda m, i, o: caps.__setitem__("addr", o.detach())),
    dec.bary_mlp.register_forward_hook(lambda m, i, o: caps.__setitem__("bary_v", o.detach())),
    dec.patch_embed.register_forward_pre_hook(lambda m, i: caps.__setitem__("dense", i[0].detach()[0])),
    dec.patch_embed.register_forward_hook(lambda m, i, o: caps.__setitem__("tok", o.detach()[0].flatten(1).T)),
    dec.norm.register_forward_hook(lambda m, i, o: caps.__setitem__("attn", o.detach())),
]
Z_F = pipe.encode(it["mesh"], it["mv_images"], it["graph"])
q0 = it["uv_queries"][0]
out = model.condition(Z_F, q0)
for h in hooks: h.remove()
H = W = dec.res
valid0 = q0["face_id"] >= 0
bmap = torch.zeros(H * W, dec.texel_dim, device=DEVICE)
bmap[valid0.reshape(-1)] = caps["bary_v"].float()
caps["bary_map"] = bmap.view(H, W, -1)

m1 = q0["valid_mask"].float()[None, None]
gg = torch.Generator(device=DEVICE).manual_seed(RANDOM_SEED)
x_t = torch.randn(1, 3, H, W, device=DEVICE, generator=gg) * m1
with torch.no_grad():
    vel = model.dit(x_t, out["uv_condition"], m1,
                    torch.tensor([700], device=DEVICE))
tex = model.generate(out["uv_condition"], q0["valid_mask"], num_steps=50,
                     seed=RANDOM_SEED)

shapes = {
    "Z_F": (tuple(Z_F.shape), (Fn, 384)),
    "face_address_table": (tuple(caps["addr"].shape), (Fn, 96)),
    "bary_address_map": (tuple(caps["bary_map"].shape), (H, W, 96)),
    "dense_texel_query": (tuple(caps["dense"].shape), (96, 256, 256)),
    "UV patch tokens": (tuple(caps["tok"].shape), (1024, 384)),
    "global_attention_out": (tuple(caps["attn"].shape), (1024, 384)),
    "uv_condition": (tuple(out["uv_condition"].shape[1:]), (64, 256, 256)),
    "fm_velocity": (tuple(vel.shape[1:]), (3, 256, 256)),
    "generated_texture": (tuple(tex.shape), (3, 256, 256)),
}
for k, (got, want) in shapes.items():
    print(f"{'PASS' if got == want else 'FAIL'} {k:22s} {got}")
    assert got == want, k

## 中间特征可视化（A–G：全部 PCA 自 live 张量）

In [ ]:
from topotex.data.mesh import CANONICAL_VIEWS, camera_matrices, rasterize_view
V3 = it["mesh"]["vertices"].cpu().numpy().astype(np.float64)
F3 = it["mesh"]["faces"].cpu().numpy().astype(np.int64)

def face_render(cols, vi, res=384):
    _, az, el = CANONICAL_VIEWS[vi]
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    img = np.ones((res, res, 3)); img[gb["mask"]] = cols[gb["face_id"][gb["mask"]]]
    return img

def pca_rgb(x):
    x = np.asarray(x, np.float64)
    xc = x - x.mean(0)
    _, _, Vt = np.linalg.svd(xc[:: max(1, len(xc) // 4096)], full_matrices=False)
    pc = xc @ Vt[:3].T
    if pc.shape[1] < 3: pc = np.pad(pc, ((0, 0), (0, 3 - pc.shape[1])))
    lo, hi = np.percentile(pc, 2, 0), np.percentile(pc, 98, 0)
    return np.clip((pc - lo) / (hi - lo + 1e-9), 0, 1)

vm_np = valid0.cpu().numpy()
panels = []
panels.append((face_render(pca_rgb(Z_F.cpu().numpy()), 0), "A. Z_F PCA on mesh"))
panels.append((face_render(pca_rgb(caps["addr"].float().cpu().numpy()), 0), "B. face address [F,96] PCA"))
bm = caps["bary_map"].cpu().numpy()
img = pca_rgb(bm.reshape(-1, 96)).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "C. bary address map [H,W,96] PCA"))
dq = caps["dense"].float().cpu().numpy()
img = pca_rgb(dq.reshape(96, -1).T).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "D. dense texel query [96,H,W] PCA"))
panels.append((pca_rgb(caps["tok"].float().cpu().numpy()).reshape(32, 32, 3), "E. UV patch tokens PCA"))
panels.append((pca_rgb(caps["attn"].float().cpu().numpy()).reshape(32, 32, 3), "F. attention output PCA"))
cx = out["uv_condition"][0].float().cpu().numpy()
img = pca_rgb(cx.reshape(64, -1).T).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "G. uv_condition PCA"))
panels.append((cx.std(0) / (cx.std(0).max() + 1e-9), "G'. uv_condition channel-std"))
fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
for ax, (im, ttl) in zip(axes.ravel(), panels):
    ax.imshow(im, interpolation="nearest" if im.shape[0] == 32 else None)
    ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 完整推理 — 同一 Z_F × 4 种 query（GT/生成/误差/渲染/seam）

In [ ]:
from topotex.data.mesh import (dilate_texture, linear_to_srgb_u8,
                               rasterize_view, render_albedo_rebake, seam_error)

def render_tex(q, tex_u8, vi, res=384):
    canon = types.SimpleNamespace(vertices=V3, faces=F3)
    uvr = types.SimpleNamespace(uv_vertices=q["uv_vertices"].astype(np.float64),
                                uv_faces=q["uv_faces"],
                                uv_face_to_mesh_face=np.arange(len(F3)))
    _, az, el = CANONICAL_VIEWS[vi]
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    v = q["valid_mask"].cpu().numpy(); t = tex_u8.copy(); t[~v] = 0
    return linear_to_srgb_u8(render_albedo_rebake(canon, uvr, dilate_texture(t, v), gb)), gb["mask"]

def psnr(a, b, m):
    mse = float(((a[m] / 255. - b[m] / 255.) ** 2).mean())
    return round(10 * np.log10(1 / max(mse, 1e-12)), 2)

ALLQ = {q["name"]: q for q in list(it["uv_queries"]) + list(it["test_uv_queries"])}
order = [("native", "uv_000"), ("xatlas", "uv_001"),
         ("blender_smart", "uv_test"), ("partial", "uv_002")]
preds, mets = {}, {}
fig, axes = plt.subplots(4, 5, figsize=(15, 12.4))
for r_i, (nm, key) in enumerate(order):
    q = ALLQ[key]
    with torch.no_grad():
        o = model.condition(Z_F, q)
    x = model.generate(o["uv_condition"], q["valid_mask"], num_steps=50, seed=RANDOM_SEED)
    im = ((x.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
    vm = q["valid_mask"].cpu().numpy(); im[~vm] = 0
    preds[key] = im
    gt = (q["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    mets[nm] = psnr(gt, im, vm)
    ra, ma = render_tex(q, im, 0); rb, mb = render_tex(q, gt, 0)
    axes[r_i][0].imshow(gt); axes[r_i][1].imshow(im)
    axes[r_i][2].imshow(np.abs(gt.astype(int) - im.astype(int)).sum(-1), cmap="inferno")
    axes[r_i][3].imshow(ra); axes[r_i][4].imshow(rb)
    axes[r_i][0].set_ylabel(f"{nm}\n{mets[nm]:.1f} dB", fontsize=10)
for c_i, t in enumerate(["GT", "generated", "|error|", "gen render", "GT render"]):
    axes[0][c_i].set_title(t, fontsize=10)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

q1 = ALLQ["uv_001"]
vm1 = q1["valid_mask"].cpu().numpy()
gt1 = (q1["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
s_gen = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], preds["uv_001"], vm1)
s_gt = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], gt1, vm1)
cons = []
for vi in range(6):
    ia, ma = render_tex(ALLQ["uv_000"], preds["uv_000"], vi)
    ib, mb = render_tex(q1, preds["uv_001"], vi)
    m = ma & mb
    if m.sum() >= 100: cons.append(psnr(ia, ib, m))
err = s_gen["per_face_error"]
cm = plt.get_cmap("inferno")(np.clip(err / max(err.max(), 1e-6), 0, 1))[:, :3]
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(face_render(cm, 0)); ax.set_title("seam heatmap (xatlas)", fontsize=9); ax.axis("off")
plt.show()
print("UV PSNR:", mets)
print(f"cross-layout consistency (native vs xatlas, 6 views): {np.mean(cons):.2f} dB")
print(f"seam: generated {s_gen['seam_error']:.4f} | GT floor {s_gt['seam_error']:.4f} "
      f"| ratio {s_gen['seam_error']/max(s_gt['seam_error'],1e-9):.2f}")
pm = ALLQ["uv_002"]["valid_mask"].cpu().numpy()
print("partial outside-mask zero:", bool((preds["uv_002"][~pm] == 0).all()))

## Flow Matching trajectory（同一次 Euler 轨迹，固定 seed）

In [ ]:
q = ALLQ[Q2DIR[QUERY]]
with torch.no_grad():
    o = model.condition(Z_F, q)
m1 = q["valid_mask"].float()[None, None]
gg = torch.Generator(device=DEVICE).manual_seed(RANDOM_SEED)
x = torch.randn(1, 3, H, W, device=DEVICE, generator=gg) * m1
taus = torch.linspace(1.0, 0.0, 51, device=DEVICE)
snaps = {1.0: x[0].clone()}
with torch.no_grad():
    for i in range(50):
        t_int = (taus[i] * model.schedule.T).round().clamp(min=1).long()
        v = model.dit(x, o["uv_condition"], m1, t_int.expand(1))
        x = (x - (taus[i] - taus[i + 1]) * v) * m1
        for tv in (0.75, 0.5, 0.25):
            if tv not in snaps and float(taus[i + 1]) <= tv + 1e-6:
                snaps[tv] = x[0].clone()
snaps[0.0] = x[0].clone()
vmq = q["valid_mask"].cpu().numpy()
def to_im(tt):
    im = ((tt.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
    im[~vmq] = 0
    return im
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, tv in zip(axes, (1.0, 0.75, 0.5, 0.25, 0.0)):
    ax.imshow(to_im(snaps[tv])); ax.set_title(f"x @ tau={tv}", fontsize=9); ax.axis("off")
axes[5].imshow((q["gt_texture"].permute(1, 2, 0).cpu().numpy()))
axes[5].set_title("GT", fontsize=9); axes[5].axis("off")
plt.suptitle(f"one Euler-50 trajectory — {QUERY} query, seed {RANDOM_SEED}", fontsize=10)
plt.tight_layout(); plt.show()

## MODE=full：随机 8 个 unseen objects 生成画廊（quick 跳过）

In [ ]:
if MODE == "full":
    rngu = np.random.default_rng(RANDOM_SEED)
    pick = [TEST[i] for i in rngu.choice(len(TEST), 8, replace=False)]
    ds_u = TopoTexDataset(DATASET_ROOT, pick, device=DEVICE)
    tiles = []
    for uit in ds_u.items:
        qq = uit["uv_queries"][0]
        z = pipe.encode(uit["mesh"], uit["mv_images"], uit["graph"])
        oo = model.condition(z, qq)
        xx = model.generate(oo["uv_condition"], qq["valid_mask"], num_steps=50, seed=RANDOM_SEED)
        imx = ((xx.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
        vmx = qq["valid_mask"].cpu().numpy(); imx[~vmx] = 0
        gtx = (qq["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        tiles += [(imx, "gen"), (gtx, "GT")]
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for i, (im, tag) in enumerate(tiles):
        ax = axes[i // 4][i % 4]; ax.imshow(im); ax.set_title(tag, fontsize=8); ax.axis("off")
    plt.suptitle("unseen objects — native query (gen|GT pairs)", fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print("quick mode: gallery skipped")